# 🧮 Day 03: Expression Evaluation — Stacks Meet Math

---

## Why Stacks for Expressions?

Think about how you evaluate `3 + 2 * 4`. You can't just go left to right because `*` has higher precedence. You need to "remember" the `3 +` part while you compute `2 * 4`.

That "remembering" is exactly what a stack does. Different expression formats just need different stack strategies:

```
Infix:    3 + 2 * 4     (human-readable, needs precedence rules)
Postfix:  3 2 4 * +     (machine-friendly, no precedence needed)
```

---

## 🎯 Problem 1: Evaluate Reverse Polish Notation (LC #150)

Postfix notation is beautifully simple: scan left to right, push numbers, and when you see an operator, pop two numbers, apply the operator, push the result.

```
Input:  ["2", "1", "+", "3", "*"]
Output: 9

Step by step:
  Push 2, Push 1
  See "+": pop 1 and 2 → 2+1 = 3 → push 3
  Push 3
  See "*": pop 3 and 3 → 3*3 = 9 → push 9
  Answer = 9
```

**Why does this work?** In postfix, operators always come right after their operands. No parentheses or precedence needed.

In [ ]:
def evalRPN(tokens):
    """
    Evaluate Reverse Polish Notation.
    Time: O(n), Space: O(n)
    """
    stack = []
    ops = {
        '+': lambda a, b: a + b,
        '-': lambda a, b: a - b,
        '*': lambda a, b: a * b,
        '/': lambda a, b: int(a / b),  # Truncate toward zero
    }
    
    for token in tokens:
        if token in ops:
            b = stack.pop()  # Second operand (popped first!)
            a = stack.pop()  # First operand
            stack.append(ops[token](a, b))
        else:
            stack.append(int(token))
    
    return stack[0]

print(evalRPN(["2", "1", "+", "3", "*"]))                        # 9
print(evalRPN(["4", "13", "5", "/", "+"]))                       # 6
print(evalRPN(["10", "6", "9", "3", "+", "-11", "*", "/",
               "*", "17", "+", "5", "+"]))                        # 22

**Watch out:** When popping for an operator, the first pop is the right operand (`b`), second pop is the left operand (`a`). Matters for subtraction and division.

---

## 🎯 Problem 2: Basic Calculator II (LC #227)

Evaluate `"3+2*2"` → 7. No parentheses, but we need to handle `+`, `-`, `*`, `/` with correct precedence.

**The trick:** Keep track of the last operator. When you see a new operator (or reach the end):
- If last op was `+` or `-`: push the number (with sign) to stack
- If last op was `*` or `/`: pop, compute immediately, push result

At the end, sum everything in the stack. This handles precedence naturally — high-precedence ops are resolved immediately, low-precedence ones are deferred.

In [ ]:
def calculate(s):
    """
    Basic Calculator II: +, -, *, / with correct precedence.
    Time: O(n), Space: O(n)
    """
    stack = []
    num = 0
    last_op = '+'  # Start with '+' so first number gets pushed
    
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)  # Build multi-digit number
        
        # Process when we hit an operator OR end of string
        if ch in '+-*/' or i == len(s) - 1:
            if last_op == '+':
                stack.append(num)
            elif last_op == '-':
                stack.append(-num)
            elif last_op == '*':
                stack.append(stack.pop() * num)
            elif last_op == '/':
                stack.append(int(stack.pop() / num))  # Truncate toward 0
            
            last_op = ch
            num = 0
    
    return sum(stack)

print(calculate("3+2*2"))    # 7  (not 10!)
print(calculate(" 3/2 "))    # 1
print(calculate(" 3+5 / 2")) # 5

**Trace for `"3+2*2"`:**
```
ch='3': num=3
ch='+': last_op='+', push 3. stack=[3], last_op='+', num=0
ch='2': num=2
ch='*': last_op='+', push 2. stack=[3,2], last_op='*', num=0
ch='2': num=2
end:    last_op='*', pop 2, push 2*2=4. stack=[3,4]
return: 3+4 = 7 ✅
```

---

## 🎯 Problem 3: Basic Calculator (LC #224)

Now with parentheses: `"(1+(4+5+2)-3)+(6+8)"` → 23. Only `+`, `-`, and `()`.

**Approach:** When we hit `(`, save our current state (result + sign) to a stack. When we hit `)`, pop that state and combine.

Think of `(` as "save context" and `)` as "restore context."

In [ ]:
def calculateI(s):
    """
    Basic Calculator I: +, -, () only.
    Time: O(n), Space: O(n)
    """
    stack = []  # Saves (result_so_far, sign_before_paren)
    result = 0
    num = 0
    sign = 1  # +1 or -1
    
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == '+':
            result += sign * num
            num = 0
            sign = 1
        elif ch == '-':
            result += sign * num
            num = 0
            sign = -1
        elif ch == '(':
            # Save current state and start fresh inside parens
            stack.append(result)
            stack.append(sign)
            result = 0
            sign = 1
        elif ch == ')':
            result += sign * num
            num = 0
            # Pop sign and previous result
            result *= stack.pop()    # sign before '('
            result += stack.pop()    # result before '('
    
    result += sign * num  # Don't forget the last number
    return result

print(calculateI("1 + 1"))                    # 2
print(calculateI(" 2-1 + 2 "))                # 3
print(calculateI("(1+(4+5+2)-3)+(6+8)"))      # 23

---

## 🗺️ Expression Evaluation Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  EXPRESSION EVALUATION PATTERNS                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. POSTFIX (RPN): Stack of numbers                                 ║
║     Number → push. Operator → pop two, compute, push result.        ║
║                                                                      ║
║  2. INFIX (no parens): Track last_op, use precedence                ║
║     High-precedence (*, /) → compute immediately (pop & push)       ║
║     Low-precedence (+, -)  → defer (just push number with sign)     ║
║     End: sum the stack.                                              ║
║                                                                      ║
║  3. INFIX (with parens): Save/restore context                       ║
║     '(' → push (result, sign), reset state                          ║
║     ')' → pop and combine with inner result                         ║
║                                                                      ║
║  GOTCHA: Division truncates toward zero → use int(a/b) not a//b     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 150 | Evaluate Reverse Polish Notation | Medium | Postfix evaluation |
| 227 | Basic Calculator II | Medium | Infix, precedence with stack |
| 224 | Basic Calculator | Hard | Infix with parentheses |
| 394 | Decode String | Medium | Nested brackets (see Module 01) |
| 772 | Basic Calculator III | Hard | All: +, -, *, /, () combined |
| 770 | Basic Calculator IV | Hard | Symbolic evaluation |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 RPN is the simplest — no precedence, just push/pop numbers     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Precedence handling: compute *, / immediately; defer +, -      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Parentheses: push context on '(', restore on ')'              │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Multi-digit: num = num * 10 + int(ch) — classic pattern       │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Always handle end-of-string as "virtual operator"             │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Queue Fundamentals

Queues, BFS connection, implementing queues with stacks, and circular queues.

---

### 🎉 Expression Evaluation = Stacks Doing Math!
If it has operators and numbers, there's a stack solution.